In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl
df_fern = df[df["Network Name"] == '-> BC-FERN']
df_pcds = df_fern[['Station ID', 'Unique Names','Unique Location (String)', 'Native ID']].reset_index(drop=True)
df_ted = df_fern[['Station_name','native_id', 'lat', 'lon', 'Elevation']].reset_index(drop=True)


df_pcds['Station_name'] = df_fern['Unique Names'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_pcds = df_pcds.drop(columns=['Unique Names'])

df_pcds['native_id'] = df_fern['Native ID'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_pcds = df_pcds.drop(columns=['Native ID'])

In [4]:
import re

def parse_lat_lon_elev(text):
    # regex to capture: lon value, W/E, lat value, N/S, elevation
    pattern = r'([\d\.]+)\s*([WE]),\s*([\d\.]+)\s*([NS]),\s*Elev\.\s*([\d\.]+)'
    m = re.search(pattern, text)
    
    if not m:
        return None, None, None

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    # Convert numbers
    lon = float(lon_val) * (-1 if lon_dir == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir == "S" else 1)
    elev = float(elev_val)

    return lat, lon, elev

# Apply to dataframe
df_pcds[['lat', 'lon', 'Elevation']] = df_pcds['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_lat_lon_elev(x))
)
df_pcds = df_pcds.drop(columns=['Unique Location (String)'])

In [5]:
# 1. Rename df_pcds columns to match df_ted
df_pcds_renamed = df_pcds.rename(columns={
    'pcds_sation_name': 'Station_name',
    'pcds_native_id': 'native_id',
    'pcds_lat': 'lat',
    'pcds_lon': 'lon',
    'pcds_elev': 'Elevation'
})

# 2. Merge on Station_name
merged = df_pcds_renamed.merge(
    df_ted,
    on='Station_name',
    how='inner',
    suffixes=('_pcds', '_ted')
)

# 3. Compare key columns
merged['lat_match']  = merged['lat_pcds']  == merged['lat_ted']
merged['lon_match']  = merged['lon_pcds']  == merged['lon_ted']
merged['elev_match'] = merged['Elevation_pcds'] == merged['Elevation_ted']
merged['id_match']   = merged['native_id_pcds'] == merged['native_id_ted']

# 4. Whether all match
merged['all_match'] = (
    merged['lat_match'] &
    merged['lon_match'] &
    merged['elev_match'] &
    merged['id_match']
)

# 5. Show results
merged[['Station_name', 'lat_match', 'lon_match', 'elev_match', 'id_match', 'all_match']]



,Station_name,lat_match,lon_match,elev_match,id_match,all_match
0,SeebachWx,True,False,True,True,False
1,Atlin School,True,True,True,True,True
2,Cassiar,True,True,True,True,True
3,Monarch-1,True,True,True,True,True
4,Northern Dancer,True,True,True,True,True
...,...,...,...,...,...,...
59,Mayson Lake - M2,True,True,True,True,True
60,Mayson Lake - M3,True,True,True,True,True
61,Mayson Lake - M4,False,False,False,True,False
62,Mayson Lake - M5,True,True,True,True,True


In [6]:
# Filter rows where not all match
mismatched = merged[merged['all_match'] == False]

# Show relevant columns
mismatched[['Station_name', 
            'lat_pcds', 'lat_ted', 
            'lon_pcds', 'lon_ted', 
            'Elevation_pcds', 'Elevation_ted', 
            'native_id_pcds', 'native_id_ted']]

df_ted.loc[df_ted['Station_name'] == 'Mayson Lake - M4', 'lon'] = '-122.058'

df_ted['Elevation'] = pd.to_numeric(df_ted['Elevation'], errors='coerce')


save_path = './output_tables/'
df_ted.to_csv(save_path + '2.1_64_fern_station_insert_info.csv', index=False)
df_ted


,Station_name,native_id,lat,lon,Elevation
0,SeebachWx,Seebach,54.355,122.058,780.0
1,Atlin School,AtlSch,59.574,-133.687,705.0
2,Cassiar,CasWx,59.251,-129.86,1577.0
3,Monarch-1,Mon1,59.544,-133.617,1410.0
4,Northern Dancer,NDWx,60.008,-131.6,1628.0
...,...,...,...,...,...
59,Mayson Lake - M2,M2,51.216528,-120.388667,1277.0
60,Mayson Lake - M3,M3,51.217611,-120.401694,1290.0
61,Mayson Lake - M4,M4,51.219111,-122.058,NaN
62,Mayson Lake - M5,M5,51.212361,-120.412222,1314.0


For the station `SeebachWx`, the lon should be `-122.058` in Ted's sheet, the left pcds part is correct

`Mayson Lake - M4` has no elev information

### Insert into db

In [7]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [8]:
from sqlalchemy import func

stations_created = []

# for _, row in df_ted.iloc[0:2].iterrows():
for _, row in df_ted.iterrows():
    
    name = row['Station_name']
    nid  = row['native_id']

    # 1. Create Station
    st = Station(
        native_id=nid,
        publish=True,
        network_id=11)


    session.add(st)
    session.flush()  # 得到 st.id

    h = History(
        station_id=st.id,
        station_name=name,
        lat=row['lat'],
        lon=row['lon'],
        elevation=row['Elevation'],
        province="BC",
        country="CA",
        the_geom=func.ST_SetSRID(func.ST_MakePoint(row['lon'], row['lat']), 4326))

    session.add(h)

    stations_created.append((name, st.id))

session.commit()

print("Created:", stations_created)

Created: [('SeebachWx', 14733), ('Atlin School', 14734), ('Cassiar', 14735), ('Monarch-1', 14736), ('Northern Dancer', 14737), ('Goose Lake Rd', 14738), ('Isobel Lake (IDFxh2)', 14739), ('LDB - Lower grassland', 14740), ('LDB - LowerUpper grassland', 14741), ('LDB - Middle grassland', 14742), ('LDB - Upper grassland', 14743), ('LDB - UpperMiddle grassland', 14744), ('Arrowstone', 14745), ('Chase Creek', 14746), ('Sicamouse Creek - WV1', 14747), ('Upper Penticton - PK', 14748), ('Upper Penticton Creek - P1', 14749), ('Upper Penticton Creek - P5', 14750), ('Upper Penticton Creek - PB', 14751), ('Upper Penticton Creek - PC', 14752), ('Upper Penticton Creek - PJ', 14753), ('Upper Penticton Creek - PL', 14754), ('CrookedLk', 14755), ('DavisPGTIS', 14756), ('Hudson Bay Stn', 14757), ('IBB1Wx', 14758), ('IBB2Wx', 14759), ('IBB3Wx', 14760), ('Inklin', 14761), ('KennedySidingCC', 14762), ('KennedySidingFor', 14763), ('SumWxCC', 14764), ('Tamarac', 14765), ('Ape Lake', 14766), ('Cain Ridge_Run',